# streaming

> Server-Sent Events (SSE) generator for real-time progress streaming to web clients.

In [ ]:
#| default_exp streaming

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, time, asyncio
from typing import Iterator, AsyncIterator
from cjm_tqdm_capture.progress_monitor import ProgressMonitor

In [ ]:
#| export
def sse_stream(
    monitor: ProgressMonitor,  # Progress monitor instance to read job updates from
    job_id: str,  # Unique identifier of the job to stream
    interval: float = 0.25,  # Polling interval in seconds for checking progress updates
    heartbeat: float = 15.0,  # Seconds between keep-alive messages when no updates
    wait_for_start: bool = True,  # Whether to wait for job to start before ending stream
    start_timeout: float = 5.0,  # Maximum seconds to wait for job to start if wait_for_start is True
) -> Iterator[str]:  # SSE-formatted strings ready to send to client
    """
    Framework-agnostic SSE generator.
    - Yields 'data: {json}\\n\\n' when progress changes.
    - Sends ': keep-alive' comments every `heartbeat` seconds when idle.
    - If `wait_for_start` is True, it will wait up to `start_timeout` for
      the first snapshot before ending (avoids race at job startup).
    """
    last = None
    start_ts = time.time()
    last_hb = start_ts

    while True:
        snap = monitor.snapshot(job_id)
        now = time.time()

        if not snap:
            if wait_for_start and (now - start_ts) < start_timeout:
                # keep the connection alive while waiting for first update
                if now - last_hb >= heartbeat:
                    yield ": waiting\n\n"
                    last_hb = now
                time.sleep(min(interval, 0.1))
                continue
            else:
                yield "event: end\ndata: {}\n\n"
                break

        latest = snap["latest"]
        changed = (
            not last
            or latest.current != getattr(last, "current", None)
            or latest.progress != getattr(last, "progress", None)
        )

        if changed:
            payload = {
                "progress": snap["overall_progress"],
                "completed": snap["completed"],
                "bars": {
                    k: {"desc": v.description, "pct": v.progress, "cur": v.current, "tot": v.total}
                    for k, v in snap["bars"].items()
                },
            }
            yield f"data: {json.dumps(payload)}\n\n"
            last = latest
            last_hb = now
        elif now - last_hb >= heartbeat:
            yield ": keep-alive\n\n"
            last_hb = now

        if snap["completed"]:
            break

        time.sleep(interval)

In [ ]:
#| export
async def sse_stream_async(
    monitor: ProgressMonitor,  # Progress monitor instance to read job updates from
    job_id: str,  # Unique identifier of the job to stream
    interval: float = 0.25,  # Polling interval in seconds for checking progress updates
    heartbeat: float = 15.0,  # Seconds between keep-alive messages when no updates
    wait_for_start: bool = True,  # Whether to wait for job to start before ending stream
    start_timeout: float = 5.0,  # Maximum seconds to wait for job to start if wait_for_start is True
) -> AsyncIterator[str]:  # SSE-formatted strings ready to send to client
    """
    Async version of SSE generator for frameworks that require async iteration.
    - Yields 'data: {json}\\n\\n' when progress changes.
    - Sends ': keep-alive' comments every `heartbeat` seconds when idle.
    - If `wait_for_start` is True, it will wait up to `start_timeout` for
      the first snapshot before ending (avoids race at job startup).
    """
    last = None
    start_ts = time.time()
    last_hb = start_ts

    while True:
        snap = monitor.snapshot(job_id)
        now = time.time()

        if not snap:
            if wait_for_start and (now - start_ts) < start_timeout:
                # keep the connection alive while waiting for first update
                if now - last_hb >= heartbeat:
                    yield ": waiting\n\n"
                    last_hb = now
                await asyncio.sleep(min(interval, 0.1))
                continue
            else:
                yield "event: end\ndata: {}\n\n"
                break

        latest = snap["latest"]
        changed = (
            not last
            or latest.current != getattr(last, "current", None)
            or latest.progress != getattr(last, "progress", None)
        )

        if changed:
            payload = {
                "progress": snap["overall_progress"],
                "completed": snap["completed"],
                "bars": {
                    k: {"desc": v.description, "pct": v.progress, "cur": v.current, "tot": v.total}
                    for k, v in snap["bars"].items()
                },
            }
            yield f"data: {json.dumps(payload)}\n\n"
            last = latest
            last_hb = now
        elif now - last_hb >= heartbeat:
            yield ": keep-alive\n\n"
            last_hb = now

        if snap["completed"]:
            break

        await asyncio.sleep(interval)

In [ ]:
#| eval: false
from fastcore.test import test_eq
import uuid, time, json
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.progress_monitor import ProgressMonitor

monitor = ProgressMonitor()
runner = JobRunner(monitor)

def slow_task():
    from tqdm import tqdm; import time
    for _ in tqdm(range(50), desc="SSE"):
        time.sleep(0.01)

jid = str(uuid.uuid4())
runner.start(jid, slow_task, patch_kwargs=dict(min_delta_pct=10, min_update_interval=0.03, emit_initial=True))

# Consume a couple of SSE chunks
gen = sse_stream(monitor, jid, interval=0.02, heartbeat=0.5)  # waits for job to start
events = []
start = time.time()
for chunk in gen:
    if chunk.startswith("data: "):
        events.append(json.loads(chunk[6:]))
        if len(events) >= 1:  # one data event is enough to prove it works
            break
    if time.time() - start > 5:
        break

assert len(events) >= 1, "No SSE data events received"
e0 = events[0]
for key in ("progress", "completed", "bars"):
    assert key in e0, f"Missing key in SSE payload: {key}"
assert isinstance(e0["bars"], dict)
print(events)

SSE:   0%|                                                                         | 0/50 [00:00<?, ?it/s]

[{'progress': 0.0, 'completed': False, 'bars': {'bar-1': {'desc': 'SSE', 'pct': 0.0, 'cur': 0, 'tot': 50}}}]


SSE: 100%|████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 98.53it/s]


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()